# Récupérer les communes d’Auvergne-Rhône-Alpes

Une **commune** est un territoire administratif possédant un code INSEE, un nom et un contour. Notre objectif est de fournir ces trois informations, ainsi que son rattachement au département et à la région, pour les communes de notre périmètre.

Choisir le noyau **Python (IGN / OSM)** dans VS Code. Exécuter les cellules dans l’ordre avec `Maj + Entrée`. Ce notebook fonctionne indépendamment du premier : il n'utilise aucune de ses variables.

Source : IGN **ADMIN EXPRESS COG**, édition du 1er janvier 2026, comme dans le premier notebook. [Présentation IGN](https://cartes.gouv.fr/aide/fr/partenaires/ign/generalites-ign/actualites/2026-06-flatgeobuf-geoparquet/).

## 1. Préparer les chemins

Nous reprenons la recherche du dossier du module. Les sources téléchargées sont dans `data/raw/ign/` ; les résultats préparés pour l'équipe seront dans `data/processed/`.

In [1]:
from pathlib import Path
from urllib.request import urlopen

import geopandas as gpd
from IPython.display import HTML, display

current_directory = Path.cwd().resolve()
module_directory = None

for directory in [current_directory, *current_directory.parents]:
    if directory.name == "ign-osm" and (directory / "requirements.txt").is_file():
        module_directory = directory
        break
    if (directory / "ign-osm" / "requirements.txt").is_file():
        module_directory = directory / "ign-osm"
        break

if module_directory is None:
    raise FileNotFoundError("Open the notebook from the project directory.")

raw_path = module_directory / "data" / "raw" / "ign" / "municipalities_2026.fgb"
output_path = module_directory / "data" / "processed" / "municipalities_84.geojson"
print(f"Module directory: {module_directory}")

Module directory: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm


## 2. Télécharger les communes

Le fichier national pèse environ **456 Mo**. La première exécution nécessite Internet et peut prendre plusieurs minutes. Les suivantes réutilisent le fichier local.

Nous copions la réponse par blocs de 1 Mo pour ne pas charger tout le téléchargement en mémoire. Un fichier temporaire `.part` évite de confondre un téléchargement interrompu avec un fichier terminé. `replace()` ne le renomme qu'une fois la récupération achevée.

La taille attendue vient du catalogue IGN ; sa vérification complète celle de la signature FlatGeobuf. Le millésime reste fixé pour travailler avec les mêmes données que le notebook précédent.

In [2]:
source_url = (
    "https://data.geopf.fr/chunk/telechargement/download/"
    "ADMIN-EXPRESS-COG-PARTIEL/"
    "ADMIN-EXPRESS-COG_4-0__FLATGEOBUF_WGS84G_FRA_2026-01-01/commune.fgb"
)
expected_size = 456183416

if raw_path.exists():
    print(f"Using existing file: {raw_path}")
else:
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = raw_path.with_suffix(".fgb.part")
    print("Downloading municipalities: approximately 456 MB...")
    with urlopen(source_url, timeout=120) as response:
        with temporary_path.open("wb") as destination:
            while chunk := response.read(1024 * 1024):
                destination.write(chunk)
    if temporary_path.stat().st_size != expected_size:
        raise ValueError("Unexpected download size; the source may have changed.")
    with temporary_path.open("rb") as source_file:
        if source_file.read(8) != b"fgb\x03fgb\x01":
            raise ValueError("The response is not a FlatGeobuf file.")
    temporary_path.replace(raw_path)

if raw_path.stat().st_size != expected_size:
    raise ValueError("The local file has an unexpected size.")
print(f"Local file size: {raw_path.stat().st_size:,} bytes")

Using existing file: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/raw/ign/municipalities_2026.fgb
Local file size: 456,183,416 bytes


## 3. Explorer les colonnes sans charger tous les contours

`rows=5` lit seulement les cinq premières lignes. Cela suffit pour découvrir le schéma, c'est-à-dire les noms et types de colonnes. `drop()` masque la géométrie pour rendre le tableau lisible.

Les colonnes de la source sont conservées en français à cette étape. Les colonnes du fichier de sortie seront en anglais.

In [3]:
sample = gpd.read_file(raw_path, rows=5)
print(list(sample.columns))
display(sample.drop(columns="geometry"))
print(f"Coordinate reference system: {sample.crs}")

['cleabs', 'nom_officiel', 'nom_officiel_en_majuscules', 'statut', 'code_insee', 'population', 'date_du_recensement', 'organisme_recenseur', 'code_insee_du_canton', 'code_insee_de_l_arrondissement', 'code_insee_du_departement', 'code_insee_de_la_region', 'code_siren', 'codes_siren_des_epci', 'code_postal', 'superficie_cadastrale', 'geometry']


,cleabs,nom_officiel,nom_officiel_en_majuscules,statut,code_insee,population,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region,code_siren,codes_siren_des_epci,code_postal,superficie_cadastrale
0,COMMUNE_0000000000097412,Saint-Joseph,SAINT-JOSEPH,Commune simple,97412,39207,2023-01-01,INSEE,97494,9742,974,04,219740123,249740085,97480,17850
1,COMMUNE_0000000000097417,Saint-Philippe,SAINT-PHILIPPE,Commune simple,97417,5082,2023-01-01,INSEE,97408,9742,974,04,219740172,249740085,97442,15390
2,COMMUNE_0000000000097419,Sainte-Rose,SAINTE-ROSE,Commune simple,97419,6557,2023-01-01,INSEE,97408,9743,974,04,219740198,249740093,97439,17760
3,COMMUNE_0000000000097406,La Plaine-des-Palmistes,LA PLAINE-DES-PALMISTES,Commune simple,97406,6936,2023-01-01,INSEE,97407,9743,974,04,219740065,249740093,97431,8320
4,COMMUNE_0000000000097422,Le Tampon,LE TAMPON,Commune simple,97422,82579,2023-01-01,INSEE,97499,9742,974,04,219740222,249740085,97430,16540


Coordinate reference system: EPSG:4326


## 4. Sélectionner les communes de la région

Le champ IGN de rattachement à la région est `code_insee_de_la_region`, visible dans le tableau précédent. Le filtre `where` est une condition SQL exécutée par le lecteur de fichier : seules les communes de la région `84` sont chargées dans notre tableau.

Nous sélectionnons des communes entières selon leur rattachement administratif. Nous ne découpons pas leurs contours sur une frontière.

In [4]:
municipalities = gpd.read_file(raw_path, where="code_insee_de_la_region = '84'")
if municipalities.empty:
    raise ValueError("No municipalities found for region 84.")
print(f"Municipality count: {len(municipalities)}")
display(municipalities.drop(columns="geometry").head())

Municipality count: 4025


,cleabs,nom_officiel,nom_officiel_en_majuscules,statut,code_insee,population,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region,code_siren,codes_siren_des_epci,code_postal,superficie_cadastrale
0,COMMUNE_0000000000007259,Saint-Just-d'Ardèche,SAINT-JUST-D'ARDECHE,Commune simple,07259,1577,2023-01-01,INSEE,0705,072,07,84,210702593,240700864,07700,1040
1,COMMUNE_0000000000026275,Rochegude,ROCHEGUDE,Commune simple,26275,1678,2023-01-01,INSEE,2614,262,26,84,212602759,200042901,26790,1830
2,COMMUNE_0000000000026357,Tulette,TULETTE,Commune simple,26357,1994,2023-01-01,INSEE,2606,262,26,84,212603575,200042901,26790,2350
3,COMMUNE_0000000000026180,Mérindol-les-Oliviers,MERINDOL-LES-OLIVIERS,Commune simple,26180,208,2023-01-01,INSEE,2610,262,26,84,212601801,200068229,26170,920
4,COMMUNE_0000000000026256,Propiac,PROPIAC,Commune simple,26256,131,2022-01-01,INSEE,2610,262,26,84,212602569,200068229,26170,1120


## 5. Préparer et vérifier la sortie

Le **code INSEE** est notre clé de jointure : il reste une chaîne de caractères. Nous conservons les codes du département et de la région pour permettre à l'équipe de regrouper les résultats.

Nous vérifions que les identifiants sont renseignés et uniques, que toutes les communes appartiennent à la région `84` et que leurs contours sont présents et valides. Une erreur arrête la cellule : nous ne réparons pas silencieusement les données.

In [5]:
municipalities = municipalities[
    ["code_insee", "nom_officiel", "code_insee_du_departement", "code_insee_de_la_region", "geometry"]
].rename(columns={
    "code_insee": "insee_code",
    "nom_officiel": "name",
    "code_insee_du_departement": "department_code",
    "code_insee_de_la_region": "region_code",
})

assert municipalities["insee_code"].notna().all()
assert municipalities["insee_code"].is_unique
assert municipalities["region_code"].eq("84").all()
assert municipalities.geometry.notna().all()
assert not municipalities.geometry.is_empty.any()
assert municipalities.geometry.is_valid.all()

municipalities = municipalities.to_crs(epsg=4326)
municipalities = municipalities.sort_values("insee_code").reset_index(drop=True)
display(municipalities.drop(columns="geometry").head(10))

,insee_code,name,department_code,region_code
0,01001,L'Abergement-Clémenciat,01,84
1,01002,L'Abergement-de-Varey,01,84
2,01004,Ambérieu-en-Bugey,01,84
3,01005,Ambérieux-en-Dombes,01,84
4,01006,Ambléon,01,84
5,01007,Ambronay,01,84
6,01008,Ambutrix,01,84
7,01009,Andert-et-Condon,01,84
8,01010,Anglefort,01,84
9,01011,Apremont,01,84


## 6. Enregistrer et relire le GeoJSON

Le résultat est enregistré puis relu pour vérifier le fichier sur disque. La source IGN reste intacte. Le GeoJSON utilise des coordonnées longitude/latitude (EPSG:4326), pas des distances en mètres.

Le tableau final indique le nombre de communes par code de département. Cette répartition est un contrôle descriptif, pas une validation indépendante du recensement officiel.

In [6]:
output_path.parent.mkdir(parents=True, exist_ok=True)
municipalities.to_file(output_path, driver="GeoJSON")
saved_municipalities = gpd.read_file(output_path)
assert len(saved_municipalities) == len(municipalities)
assert set(saved_municipalities["insee_code"]) == set(municipalities["insee_code"])
assert saved_municipalities.geometry.is_valid.all()
print(f"Saved and verified {len(saved_municipalities)} municipalities: {output_path}")
display(saved_municipalities.groupby("department_code").size().rename("municipality_count").to_frame())

Saved and verified 4025 municipalities: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/municipalities_84.geojson


,municipality_count
department_code,
01,391
03,317
07,335
15,250
26,362
38,512
42,320
43,257
63,463


## 7. Afficher les contours des communes

Nous dessinons toutes les communes pour voir leur découpage à l'intérieur de la région. La projection Lambert-93 (`EPSG:2154`) exprime les coordonnées en mètres. La simplification à 100 mètres allège seulement l'aperçu ; le GeoJSON exporté reste inchangé.

Chaque commune fournit un dessin SVG. Nous les réunissons dans une même page HTML, affichée ci-dessous et enregistrée pour une consultation hors connexion. Les traits représentent les limites administratives, pas les rues.

Les contours sont simplifiés séparément : de petits écarts visuels peuvent apparaître entre communes voisines. Cet aperçu ne sert pas à mesurer une distance ou à vérifier la précision topologique des frontières.


In [7]:
preview_municipalities = saved_municipalities.to_crs(epsg=2154)
boundaries = preview_municipalities.geometry.simplify(100, preserve_topology=True)
min_x, min_y, max_x, max_y = boundaries.total_bounds
padding = 10000
width = max_x - min_x + 2 * padding
height = max_y - min_y + 2 * padding
view_box = f"{min_x - padding} {-max_y - padding} {width} {height}"

boundary_svg = "\n".join(
    boundary.svg(fill_color="#dbeafe", opacity=1)
    for boundary in boundaries
)
preview_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Auvergne-Rhône-Alpes municipalities</title>
  <style>
    .municipality-preview {{ font-family: sans-serif; max-width: 1100px; margin: 24px auto; padding: 16px; }}
    .municipality-preview svg {{ width: 100%; height: 75vh; background: #f8fafc; }}
    .municipality-preview path {{ stroke: #334155; stroke-width: 110; }}
  </style>
</head>
<body>
  <div class="municipality-preview">
    <h1>Auvergne-Rhône-Alpes municipalities</h1>
    <p>{len(saved_municipalities):,} municipalities | North is up | IGN ADMIN EXPRESS COG 2026</p>
    <svg xmlns="http://www.w3.org/2000/svg" viewBox="{view_box}"
         role="img" aria-label="Municipal boundaries of Auvergne-Rhône-Alpes">
      <g transform="scale(1,-1)">{boundary_svg}</g>
    </svg>
    <p>Display simplified to 100 metres. Original GeoJSON unchanged.</p>
    <p>Administrative boundaries only: no roads or population data.</p>
  </div>
</body>
</html>
"""
preview_path = module_directory / "data" / "processed" / "municipalities_84_preview.html"
preview_path.write_text(preview_html, encoding="utf-8")
print(f"Saved preview: {preview_path}")
display(HTML(preview_html))

Saved preview: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/municipalities_84_preview.html


## Bilan

Le fichier `data/processed/municipalities_84.geojson` contient les communes de la région avec leurs codes et contours. Il permettra de rattacher les résultats à une commune ou de les regrouper par département.

Le fichier source contient aussi une population agrégée par commune et sa date de recensement. Nous ne l’exportons pas ici : elle ne suffit pas pour estimer les habitants accessibles à cinq minutes à pied. Les données fines de population seront préparées séparément. Ces contours ne décrivent pas les rues. Un aperçu est affiché ci-dessus et enregistré dans `data/processed/municipalities_84_preview.html`. Le réseau OSM sera abordé ensuite.